In [1]:
import numpy as np
import pandas as pd
import scanpy as sc

from format_adata import (
    format_adata,
    filter_adata
)
from plot_Hh_ligands import (
    plot_UMAP_custom,
    calculate_vmin_vmax,
    plot_Hh_source_feature_plot,
    plot_Hh_source_dot_plot,
    plot_stacked_violins
)

import warnings
warnings.filterwarnings('ignore')

#### Define data paths and file names, and figure output path

In [2]:
adata_dir = "./public_datasets/002_PijuanSala2019/"
adata_fname = "gastrulation_ps2019.h5ad"
metadata_path = "./3-PublicDatasets/12-MouseGastrulation_Pijuan2019/MouseGastrulation-Pijuan2019_UMAPcoord-Colors.csv"
fig_output_path = "./projects_python/Hh_ligands_Pijuan_data/paper_plots/"

#### Format and save adata

In [3]:
adata_formatted = format_adata(adata_dir, adata_fname, metadata_path)

#### Filter and save adata

In [4]:
adata_filtered = filter_adata(adata_formatted, adata_dir)

In [5]:
del adata_formatted

#### Separate adata into dict by stage for plotting stacked violins (facilitates getting info by stage). Can also use this dict to plot UMAP and feature plot by stage

In [ ]:
# get list of stages
stage_list = [str(item) for item in np.unique(list(adata_filtered.obs["stage"]))]

#separate gastr_ext into list of adata by stage
adata_by_stage = [adata_filtered[(adata_filtered.obs.stage == stage), :] for
                    stage in stage_list]

adata_stages_dict = dict(zip(stage_list, adata_by_stage))

#### Plot UMAP with all stages combined

In [ ]:
umap_coords_obsm = "UMAP_coordinates"
palette = adata_formatted.uns["color_dict"]
umap_fname = "UMAP_celltypes.svg"
fig_size = (5, 3)
fig_title = "Pijuan-Sala 2019 Atlas"

plot_UMAP_custom(adata_formatted, umap_coords_obsm, palette,
                 umap_fname, fig_size, fig_title, 
                 fig_output_path)

#### Plot feature plot for each Hh ligand

In [ ]:
### calculate vmin and vmax for plotting
Hh_sources = ["Shh", "Ihh", "Dhh"]
vmin, vmax = calculate_vmin_vmax(adata_formatted, Hh_sources)

In [ ]:
umap_coords_obsm = "UMAP_coordinates"
fig_size = (3, 2.75)
#loop through list of Hh ligands
for Hh_ligand in Hh_sources:
    plot_Hh_source_feature_plot(adata_formatted, Hh_ligand, 
        umap_coords_obsm, fig_size, vmin, vmax,
        fig_output_path)

#### Plot dot plot for each Hh ligand (with and without 10% expression threshold)

In [ ]:
Hh_sources = ["Shh", "Ihh", "Dhh"]
#with expression threshold, only plot Shh and Ihh (Dhh has too low of expression)
plot_Hh_source_dot_plot(adata_filtered, Hh_sources[:2], 2.2,
    fig_output_path, exp_threshold=10, annotate_frac_exp=True)

In [ ]:
#without expression threshold, plot all 3 ligands (not shown in paper)
Hh_sources = ["Shh", "Ihh", "Dhh"]
plot_Hh_source_dot_plot(adata_filtered, Hh_sources, 12,
    fig_output_path, exp_threshold=None, annotate_frac_exp=False)

#### Plot stacked violin for each Hh ligand

In [ ]:
Hh_sources = ["Shh", "Ihh", "Dhh"]
plot_stacked_violins(adata_stages_dict, Hh_sources, fig_output_path)